# B — Triples to graph (CPU)

Everything downstream of extraction. Reads notebook A's `biology_all_triples.csv`, builds
a node/edge graph, and exports it for Neo4j and the verifier. No GPU — iterate freely.

**Two settings at the top are research decisions, not defaults to accept blindly:**
`DROP_RELATIONS` (which relation types are too unreliable to keep) and `MENTION_EDGE_TYPE`
(whether entity mention-links inherit the original relation or get a neutral one). Both
are explained where they appear.

In [ ]:
import re, glob, collections
from pathlib import Path
import pandas as pd

# ---- decisions ----------------------------------------------------------
# Relations the quality review shows are unreliable. Empty until that review exists —
# do not guess. 'অবস্থান' is the current suspect (see quality_sample.csv).
DROP_RELATIONS = []

# How to type an edge recovered by finding entity B mentioned inside the object text of
# a triple about entity A. Inheriting the original relation asserts things the textbook
# did not say — "মূল --[অংশ]--> খাদ্য" from "খাদ্য জমা থাকে" claims food is PART OF a root.
# A neutral type keeps the connectivity without the false claim. None = inherit.
MENTION_EDGE_TYPE = "সম্পর্কিত"
# -------------------------------------------------------------------------

hits = (glob.glob("/kaggle/input/**/biology_all_triples.csv", recursive=True)
        or glob.glob("kg/triples/biology_all_triples.csv")
        or glob.glob("../kg/triples/biology_all_triples.csv"))
if not hits:
    raise SystemExit("biology_all_triples.csv not found — attach notebook A's output.")

t = pd.read_csv(hits[0])
t["subject"] = t.subject.astype(str).str.strip()
t["object"] = t.object.astype(str).str.strip()
print(f"{len(t)} triples, {t.chapter_no.nunique()} chapters, {t.relation.nunique()} relations")

if DROP_RELATIONS:
    before = len(t)
    t = t[~t.relation.isin(DROP_RELATIONS)].reset_index(drop=True)
    print(f"dropped {before - len(t)} triples from {DROP_RELATIONS}")

## 1 — Canonical surface forms

Only whitespace and punctuation are normalised. Aggressive Bangla suffix stripping was
measured and rejected: it merged 14 of 1,121 subjects while risking real distinctions,
so inflection is not what makes this graph sparse.

In [ ]:
WORD = re.compile(r"[ঀ-৿]+|[A-Za-z0-9]+")


def canon(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s.strip(" ।,:;()[]\"'-–—")


t["subject"] = t.subject.map(canon)
t["object"] = t.object.map(canon)
t = t[(t.subject.str.len() > 1) & (t.object.str.len() > 1)].reset_index(drop=True)

# A subject should be a noun phrase. A long one is usually a clause the extractor
# mis-assigned — "ইচ্ছানুযায়ী সংকুচিত বা প্রসারিত হয়" appeared as a subject. Flagged,
# not dropped: the review decides whether the cutoff is right.
t["subj_tokens"] = t.subject.map(lambda s: len(WORD.findall(s)))
t["subj_suspect"] = t.subj_tokens > 5
print(f"subjects longer than 5 tokens (likely clauses): {t.subj_suspect.sum()} "
      f"({t.subj_suspect.mean()*100:.1f}%)")
t[t.subj_suspect].subject.head(5).to_list()

## 2 — Nodes

An entity is any canonical subject. Objects are kept as literal values on the fact edge
rather than promoted to nodes — most are descriptive phrases, not entities.

In [ ]:
ent = sorted(set(t.subject))
nodes = pd.DataFrame({"node_id": range(len(ent)), "label": ent})
nid = dict(zip(nodes.label, nodes.node_id))

chap = t.groupby("subject").chapter_no.agg(lambda s: sorted(set(s)))
deg = t.subject.value_counts()
nodes["chapters"] = nodes.label.map(lambda l: ",".join(map(str, chap.get(l, []))))
nodes["n_facts"] = nodes.label.map(lambda l: int(deg.get(l, 0)))

print(f"{len(nodes)} entity nodes")
print(f"appearing in >1 chapter: {(nodes.chapters.str.contains(',')).sum()}")
nodes.sort_values('n_facts', ascending=False).head(8)

## 3 — Fact edges

One per triple: the literal claim the textbook makes. These are what the verifier checks
a tutor's answer against.

In [ ]:
facts = pd.DataFrame({
    "triple_id": t.triple_id,
    "head_id": t.subject.map(nid),
    "head": t.subject,
    "relation": t.relation,
    "value": t.object,
    "chapter_no": t.chapter_no,
    "chunk_id": t.chunk_id,
    "subj_suspect": t.subj_suspect,
})
print(f"{len(facts)} fact edges")
print(facts.relation.value_counts().to_string())

## 4 — Mention edges (entity ↔ entity)

Whole-token matching, so `তন্দ্র` cannot match inside `টিস্যুতন্দ্র`. This is what gives the
graph paths to traverse; without it, 93% of facts are dead ends.

In [ ]:
SUFFIXES = ["গুলোর", "গুলোকে", "গুলো", "টিকে", "গুলি", "দের", "টির", "টি",
            "য়ের", "এর", "কে", "ের", "রা", "র"]
GENERIC = {"মাধ্যম", "ধরন", "জিনিস", "সময়", "স্থান", "গুরুত্ব", "নাম",
           "ব্যাপার", "কারণ", "ফলে", "দিক"}


def lemma(w):
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[: -len(s)]
    return w


def toks(s):
    return [lemma(w) for w in WORD.findall(s)]


vocab = {e: tuple(toks(e)) for e in ent}
vocab = {e: tk for e, tk in vocab.items()
         if tk and len("".join(tk)) >= 4 and e not in GENERIC}
by_len = collections.defaultdict(dict)
for e_, tk in vocab.items():
    by_len[len(tk)][tk] = e_
max_n = max(by_len)


def mentions(text):
    tk, found, covered = toks(text), [], set()
    for n in range(max_n, 0, -1):
        table = by_len.get(n)
        if not table:
            continue
        for i in range(len(tk) - n + 1):
            if any(j in covered for j in range(i, i + n)):
                continue
            hit = table.get(tuple(tk[i:i + n]))
            if hit:
                found.append(hit)
                covered.update(range(i, i + n))
    return found


rows = []
for r in t.itertuples():
    for tail in mentions(r.object):
        if tail == r.subject:
            continue
        rows.append({
            "triple_id": r.triple_id,
            "head_id": nid[r.subject], "head": r.subject,
            "tail_id": nid[tail], "tail": tail,
            "relation": MENTION_EDGE_TYPE or r.relation,
            "source_relation": r.relation,
            "chapter_no": r.chapter_no,
        })

ment = pd.DataFrame(rows).drop_duplicates(subset=["head", "tail", "relation"])
linked_triples = ment.triple_id.nunique()
print(f"{len(ment)} mention edges over {linked_triples} triples "
      f"({linked_triples/len(t)*100:.0f}% of facts now connect to another entity)")
print(f"edge type: {MENTION_EDGE_TYPE or 'inherited from source relation'}")

## 5 — Is it actually a graph?

The proposal's verification method relies on path-consistency, which needs paths. A graph
that is mostly isolated nodes cannot support it, so this is the number that decides whether
the method as written is viable.

In [ ]:
adj = collections.defaultdict(set)
for r in ment.itertuples():
    adj[r.head_id].add(r.tail_id)
    adj[r.tail_id].add(r.head_id)

seen, comps = set(), []
for n in nodes.node_id:
    if n in seen:
        continue
    stack, comp = [n], []
    seen.add(n)
    while stack:
        c = stack.pop()
        comp.append(c)
        for nb in adj[c]:
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    comps.append(comp)

comps.sort(key=len, reverse=True)
isolated = sum(1 for c in comps if len(c) == 1)
print(f"nodes: {len(nodes)}   mention edges: {len(ment)}")
print(f"connected components: {len(comps)}")
print(f"largest component: {len(comps[0])} nodes ({len(comps[0])/len(nodes)*100:.0f}%)")
print(f"isolated nodes: {isolated} ({isolated/len(nodes)*100:.0f}%)")
print(f"component sizes (top 8): {[len(c) for c in comps[:8]]}")

## 6 — Export

In [ ]:
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("kg/graph")
OUT.mkdir(parents=True, exist_ok=True)

nodes.to_csv(OUT / "nodes.csv", index=False)
facts.to_csv(OUT / "fact_edges.csv", index=False)
ment.to_csv(OUT / "mention_edges.csv", index=False)
print(f"wrote {len(nodes)} nodes, {len(facts)} fact edges, {len(ment)} mention edges -> {OUT}")